In [2]:
import gc
import json
from pathlib import Path

import torch
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

from utils import compute_metrics, init_model, load_splits, tokenize_dataset


In [3]:
#global random seed
SEED = 42
torch.manual_seed(SEED)

In [4]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


device(type='cuda')

In [5]:
X_train, X_val, X_test, y_train, y_val, y_test = load_splits()

In [6]:
def training_loop(ds_train, ds_val, learning_rate, model_name, per_device_train_batch_size,
                  gradient_accumulation_step, seed=SEED):
    print(f"Training with learning rate: {learning_rate}")
    model = trainer = None
    training_args = TrainingArguments(
        output_dir=f"models/{model_name}_{learning_rate}",
        learning_rate=learning_rate,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=gradient_accumulation_step,
        num_train_epochs=5,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=5,
        logging_strategy="epoch",
        report_to="none",
        seed=seed,
        data_seed=seed,
    )
    try:
        # Seed Python, NumPy, and Torch before the randomly initialized classifier head.
        set_seed(seed)
        model = init_model(model_name)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=ds_train,
            eval_dataset=ds_val,
            processing_class=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer),
            compute_metrics=compute_metrics,
        )
        trainer.train()
        # Persist the final TrainerState, including the full history and training summary.
        trainer.save_state()
        return Path(training_args.output_dir)
    finally:
        trainer = model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


training loop below, this was reused 

In [16]:
model = "bert-base-uncased"

per_device_train_batch_size=32

gradient_accumulation_step=1

for learning_rate in [4e-5]:
    ds_train = tokenize_dataset(X_train, y_train, model)
    ds_val =  tokenize_dataset(X_val, y_val, model)     
    training_loop(ds_train, ds_val, learning_rate,
                model, per_device_train_batch_size, gradient_accumulation_step, seed=SEED)



Map: 100%|██████████| 9196/9196 [00:00<00:00, 16030.08 examples/s]


Training with learning rate: 4e-05


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1021.64it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Log Loss,Runtime,Samples Per Second,Steps Per Second
1,0.283855,0.225148,0.903001,0.904944,0.898435,0.900961,0.225149,9.483400,969.691000,30.369000
2,0.143830,0.265235,0.907242,0.905181,0.908905,0.906467,0.265238,9.193300,1000.297000,31.327000
3,0.065835,0.400262,0.911375,0.910785,0.909337,0.910013,0.400255,10.770700,853.796000,26.739000
4,0.027222,0.490261,0.912027,0.910741,0.910969,0.910854,0.490236,9.986200,920.868000,28.840000
5,0.010097,0.614620,0.912027,0.911817,0.909589,0.910597,0.614548,10.543100,872.225000,27.316000


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.32s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

In [17]:
model_names = ["bert-base-uncased"]
learning_rates = [4e-5]
Path("logs").mkdir(exist_ok=True)

for model_name in model_names:
    for learning_rate in learning_rates:
        state_path = Path("models") / f"{model_name}_{learning_rate}" / "trainer_state.json"
        if not state_path.is_file():
            continue
        history = json.loads(state_path.read_text())["log_history"]
        training_by_epoch = {
            entry["epoch"]: entry for entry in history if "loss" in entry
        }
        results = []
        for entry in history:
            if "eval_f1_macro" in entry:
                results.append({
                    **training_by_epoch.get(entry["epoch"], {}),
                    **entry,
                    "learning_rate": learning_rate,
                })
        log_path = Path("logs") / f"{model_name}_{learning_rate}.json"
        log_path.write_text(json.dumps(results, indent=4))
        print(f"Wrote {log_path}")


Wrote logs/bert-base-uncased_4e-05.json
